In [1]:
from sentence_transformers import SentenceTransformer
import faiss

works_classifier = [
  {
    "id": 1,
    "name": "Разработка фронтенда"
  },
  {
    "id": 2,
    "name": "Разработка бэкенда"
  },
  {
    "id": 3,
    "name": "Тестирование ПО"
  },
  {
    "id": 4,
    "name": "Администрирование баз данных"
  },
  {
    "id": 5,
    "name": "Проектирование архитектуры"
  },
  {
    "id": 6,
    "name": "Техническая поддержка пользователей"
  },
  {
    "id": 7,
    "name": "Настройка CI/CD"
  },
  {
    "id": 8,
    "name": "Разработка мобильных приложений"
  },
  {
    "id": 9,
    "name": "Аналитика данных"
  },
  {
    "id": 10,
    "name": "Дизайн пользовательских интерфейсов"
  }
]

model = SentenceTransformer("cointegrated/rubert-tiny2")

classifier_texts = [j["name"] for j in works_classifier]
classifier_embeddings = model.encode(classifier_texts, convert_to_numpy=True, normalize_embeddings=True)

/Users/ilyachurakov/Documents/Personal/Projects/hackatons/case-07-construction-site-monitoring-team-25/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 5989.37it/s]
[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weig

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import faiss

works_list = [
  "Фронтенд-разработка",
  "Разработка frontend",
  "Бэкенд разработка",
  "Backend developer",
  "Тестирование программного обеспечения",
  "QA-инженер",
  "Админ баз данных",
  "Администрирование БД",
  "Архитектор ПО",
  "Проектирование архитектуры системы",
  "Техподдержка пользователей",
  "Support инженеров",
  "Настройка CI / CD",
  "DevOps: пайплайны и деплой",
  "Разработка мобильных приложений под iOS и Android",
  "Мобильная разработка",
  "Анализ данных",
  "Data analyst",
  "Дизайн интерфейсов",
  "UI/UX дизайн",
  "Системное администрирование",
  "Маркетинг в интернете",
  "Продажи B2B",
  "Бухгалтерский учёт"
]
print('0')
input_embeddings = model.encode(works_list, convert_to_numpy=True, normalize_embeddings=True)
print('1')
classifier_embeddings = classifier_embeddings.astype("float32")
input_embeddings = input_embeddings.astype("float32")
print('2')
dim = classifier_embeddings.shape[1]
print('3')
M = 16
index = faiss.IndexHNSWFlat(dim, M)
print('4')
index.hnsw.efConstruction = 40
print('5')

print("classifier_embeddings.shape:", classifier_embeddings.shape)
print("classifier_embeddings.dtype:", classifier_embeddings.dtype)
print("dim:", dim)
print("M:", M)
print("index:", index)

index.add(classifier_embeddings)
print('6')
k = 1

index.hnsw.efSearch = 32

D, I = index.search(input_embeddings, k)

THRESHOLD = 0.75

mapping = []
for i, job_name in enumerate(works_list):
    best_idx = int(I[i, 0])
    best_score = float(D[i, 0])

    mapping.append({
        "input_job": job_name,
        "classifier_job_id": works_classifier[best_idx]["id"] if best_score >= THRESHOLD else None,
        "classifier_job_name": works_classifier[best_idx]["name"] if best_score >= THRESHOLD else None,
        "score": best_score,
    })

for m in mapping:
    print(m)